# Pilot runs — slm-audio-evidence
Runtime → Change runtime type → **T4 GPU**. Then run cells top to bottom.
Before cell 4: upload `pilot_audio.zip` via the Files panel (left sidebar).

In [ ]:
!nvidia-smi -L

In [ ]:
!git clone https://github.com/ladnlav/slm-audio-evidence.git
%cd slm-audio-evidence

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# upload pilot_audio.zip to /content first (drag & drop into the Files panel)
!unzip -q -o /content/pilot_audio.zip -d .
!ls data/audio/spoken_squad_test | head -3
!python -c "import json,sys; rows=[json.loads(l) for l in open('data/manifests/pilot.jsonl',encoding='utf-8')]; import os; miss=[r['id'] for r in rows if not os.path.exists(r['audio_path'])]; print(len(rows),'items,',len(miss),'missing audio'); print(miss[:5])"

In [ ]:
# Run 1 (the headline): Qwen2-Audio, plain prompt. First run also downloads the weights (~15-30 min).
!python -m src.inference --model qwen2audio --strategy plain --data data/manifests/pilot.jsonl --out results/

In [ ]:
# Run 2: Qwen2-Audio, IDK prompt
!python -m src.inference --model qwen2audio --strategy s1_idk --data data/manifests/pilot.jsonl --out results/

In [ ]:
# IMPORTANT before runs 3-4: free VRAM from Qwen2-Audio — Runtime → Restart runtime,
# then re-run cells 2 (cd) and 4 (unzip) only, and continue here.
!python -m src.inference --model cascade --strategy plain --data data/manifests/pilot.jsonl --out results/

In [ ]:
# Run 4: cascade, IDK prompt
!python -m src.inference --model cascade --strategy s1_idk --data data/manifests/pilot.jsonl --out results/

In [ ]:
# Zip and download everything produced so far (run after EACH finished run — don't wait for all four)
!zip -q -r results_runs.zip results -x '*.gitkeep'
from google.colab import files
files.download('results_runs.zip')